# SignalForge Founder Command Center

Use this notebook as your founder command room for the project. Run the code cells first, read the current truth, then update the review cells with what changed and what you decided.

## How To Use It
- Run the code cells at the start of every work session.
- Read the generated founder snapshot before you change anything.
- Write down the current bottleneck, your decision, and your next move.
- Use this notebook for truth, not hope.

# SignalForge Founder Command Center

Use this notebook as the single founder review surface for the project. It is designed for one job: help you see the current truth, the current bottleneck, and the next decision without guessing.

## How To Use This Notebook
- Run the first 4 code cells at the start of each working session.
- Read the generated founder snapshot before you do any new work.
- Update the markdown log cells with what changed, what you decided, and what matters next.
- Keep this notebook focused on reality, not ideas. Use it to track execution, bottlenecks, and founder decisions.

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import Markdown, display

def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / 'fund_data').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not find SignalForge repo root from the current notebook path.')

ROOT = find_repo_root()
FUND = ROOT / 'fund_data'

def load_json(name: str, default):
    path = FUND / name
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text())
    except Exception:
        return default

def load_jsonl(name: str):
    path = FUND / name
    if not path.exists():
        return []
    rows = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            payload = json.loads(line)
        except Exception:
            continue
        if isinstance(payload, dict):
            rows.append(payload)
    return rows

def safe_list(value):
    return value if isinstance(value, list) else []

def top_reasons(reasons, limit: int = 5):
    cleaned = [str(reason).strip() for reason in safe_list(reasons) if str(reason).strip()]
    return cleaned[:limit]

In [ ]:
live_state = load_json('live_state.json', {})
paper_validation = load_json('paper_validation_status.json', {})
health = load_json('health.json', {})
deployment_gate = load_json('deployment_gate_status.json', {})
certification = load_json('production_certification_status.json', {})
market_snapshot = load_json('market_snapshot.json', {})
trade_journal = load_json('trade_journal.json', [])
live_trade_ledger = load_json('live_trade_ledger.json', [])
adaptive_ledger = load_jsonl('adaptive_cycle_ledger.jsonl')

adaptive = live_state.get('adaptive_cycle') or {}
latest_cycle = adaptive_ledger[-1] if adaptive_ledger else {}
latest_risk_events = [row for row in safe_list(live_trade_ledger) if row.get('entry_type') == 'risk_event'][-5:]

founder_snapshot = {
    'repo_root': str(ROOT),
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'stage': 'paper-validation',
    'capital': live_state.get('capital'),
    'iteration': live_state.get('iteration'),
    'trade_counter': live_state.get('trade_counter'),
    'open_positions': len(safe_list(live_state.get('open_positions'))),
    'closed_count': live_state.get('closed_count'),
    'adaptive_safety_action': adaptive.get('safety_action'),
    'adaptive_safety_reasons': top_reasons(adaptive.get('safety_reasons')),
    'target_volatility': adaptive.get('target_volatility'),
    'realized_volatility': adaptive.get('realized_volatility'),
    'smoothed_volatility': adaptive.get('smoothed_volatility'),
    'tracking_error': adaptive.get('volatility_tracking_error'),
    'paper_validation_ready': paper_validation.get('ready_for_live'),
    'paper_cycles': paper_validation.get('cycle_count'),
    'paper_trades': paper_validation.get('trade_count'),
    'latest_validation_safety_action': paper_validation.get('latest_safety_action'),
    'validation_blockers': top_reasons(paper_validation.get('reasons')),
    'health_status': health.get('overall_status'),
    'health_halt_reason': health.get('halt_reason'),
    'deployment_gate': deployment_gate.get('decision') or deployment_gate.get('confirmation'),
    'certification_ready': certification.get('ready_for_live'),
    'latest_cycle_timestamp': latest_cycle.get('timestamp'),
}

founder_snapshot

In [ ]:
stage_label = 'Paper Validation'
if founder_snapshot['paper_validation_ready']:
    stage_label = 'Ready For Controlled Live Gate'
elif founder_snapshot['adaptive_safety_action'] not in {None, 'allow'}:
    stage_label = 'Paper Validation / Adaptive Gating Bottleneck'
elif not founder_snapshot['paper_trades']:
    stage_label = 'Paper Validation / No-Trade Bottleneck'

blockers = founder_snapshot['validation_blockers'] or founder_snapshot['adaptive_safety_reasons']
risk_lines = []
for row in latest_risk_events[-3:]:
    risk_lines.append(f"- {row.get('timestamp', 'unknown time')}: {row.get('risk_details', 'no details')}")
if not risk_lines:
    risk_lines = ['- No recent risk-event entries recorded.']

snapshot_md = f"""
# Founder Snapshot

## Current Stage
- {stage_label}
- Checked at: {founder_snapshot['checked_at_utc']}

## Operating Truth
- Capital: {founder_snapshot['capital']}
- Iteration: {founder_snapshot['iteration']}
- Trade counter: {founder_snapshot['trade_counter']}
- Open positions: {founder_snapshot['open_positions']}
- Closed trades: {founder_snapshot['closed_count']}
- Adaptive safety action: {founder_snapshot['adaptive_safety_action']}
- Latest validation safety action: {founder_snapshot['latest_validation_safety_action']}
- Paper ready for live: {founder_snapshot['paper_validation_ready']}
- Paper cycles: {founder_snapshot['paper_cycles']}
- Paper trades: {founder_snapshot['paper_trades']}

## Risk + Health
- Health status: {founder_snapshot['health_status']}
- Health halt reason: {founder_snapshot['health_halt_reason']}
- Deployment gate: {founder_snapshot['deployment_gate']}
- Certification ready: {founder_snapshot['certification_ready']}

## Adaptive Metrics
- Target volatility: {founder_snapshot['target_volatility']}
- Realized volatility: {founder_snapshot['realized_volatility']}
- Smoothed volatility: {founder_snapshot['smoothed_volatility']}
- Tracking error: {founder_snapshot['tracking_error']}

## Top Blockers
{chr(10).join('- ' + reason for reason in blockers) if blockers else '- No blockers recorded in the latest artifacts.'}

## Recent Risk Events
{chr(10).join(risk_lines)}
"""
display(Markdown(snapshot_md))

In [ ]:
suggestions = []
if founder_snapshot['adaptive_safety_action'] not in {None, 'allow'}:
    suggestions.append('Resolve the adaptive gating reason before expecting paper entries to advance.')
if founder_snapshot['paper_trades'] in {None, 0}:
    suggestions.append('Prioritize understanding why valid raw signals are not appearing often enough.')
if founder_snapshot['paper_cycles'] is not None and founder_snapshot['paper_cycles'] < 100:
    suggestions.append('Keep accumulating clean paper cycles; the live gate cannot open with low runtime evidence.')
if founder_snapshot['deployment_gate'] and str(founder_snapshot['deployment_gate']).upper() != 'GO':
    suggestions.append('Treat deployment gate status as a founder-level blocker, not a minor ops issue.')
if not suggestions:
    suggestions.append('System looks operational. Focus on scaling evidence and narrowing the next milestone.')

next_moves_md = '## Founder Next Moves\n' + '\n'.join(f'- {item}' for item in suggestions)
display(Markdown(next_moves_md))

# Daily Founder Log

## Date
-

## What Changed Today
-

## Current Bottleneck
-

## What I Decided
-

## What I Will Do Next
-

## What I Should Stop Doing
-

# Weekly Founder Review

## This Week's Outcome
- Main objective:
- Outcome achieved:
- Proof from artifacts:

## Business Reality
- What is actually working:
- What is still blocked:
- What would make this business stronger next week:

## Product Reality
- Paper trading status:
- Live-readiness status:
- Biggest product risk:

## Founder Decisions
- Continue:
- Cut:
- Double down:

## Next Week Top 3
1.
2.
3.

# Investor Narrative Notes

Use this cell to keep your founder story honest and up to date.

## Why This Company Exists
-

## What We Know That Others Miss
-

## Proof We Have Today
-

## What Must Be True In The Next 30 Days
-